# Disease-to-Target Protein Discovery Agent

Research-support prototype that uses PubMed literature and UniProtKB protein records to explore disease-to-target hypotheses. See README.md for setup and usage.

**Research use only:** outputs are hypotheses for expert review, not medical advice or validated treatment targets.


In [ ]:
### Autonomous Disease to Target Agent
import os
import json
from typing import Any

import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
load_dotenv()


In [ ]:
NCBI_BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/search"

In [ ]:
###NCBI Parameters
def _ncbi_parameters() -> dict[str, str]:
    """Return shared NCBI parameters, including an email when configured."""
    parameters = {"tool": "disease_to_target_agent"}
    if email := os.getenv("NCBI_EMAIL"):
        parameters["email"] = email
    return parameters

In [ ]:
### Pubmed Tool
@tool
def search_pubmed(query: str, max_results: int = 5) -> str:
    """Search PubMed and return recent matching papers with their PMIDs and links.
    Use this first when assessing a disease or proposed biological target.
    """
    try:
        # First request: find PubMed IDs
        search_response = requests.get(
            f"{NCBI_BASE_URL}/esearch.fcgi",
            params={
                **_ncbi_parameters(),
                "db": "pubmed",
                "term": query,
                "retmode": "json",
                "retmax": max_results,
                "sort": "date",
            },
            timeout=20,
        )

        search_response.raise_for_status()
        identifiers = search_response.json()["esearchresult"].get("idlist", [])

        if not identifiers:
            return "No PubMed records found for the query."

        # Second request: fetch details for those IDs
        summary_response = requests.get(
            f"{NCBI_BASE_URL}/esummary.fcgi",
            params={
                **_ncbi_parameters(),
                "db": "pubmed",
                "id": ",".join(identifiers),
                "retmode": "json",
            },
            timeout=20,
        )
## call back of paper with pmid, titled-untitled, pub date
        summary_response.raise_for_status()
        records: dict[str, Any] = summary_response.json()["result"]

        articles = [
            {
                "pmid": pmid,
                "title": records[pmid].get("title", "Untitled"),
                "journal": records[pmid].get(
                    "fulljournalname", "Unknown journal"
                ),
                "publication_date": records[pmid].get(
                    "pubdate", "Unknown date"
                ),
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            }
            for pmid in identifiers
        ]

        return json.dumps(articles, indent=2)

    except requests.RequestException as error:
        return f"PubMed search failed: {error}. Do not invent literature evidence."

In [ ]:
### PubMed abstract reading tool
@tool
def fetch_pubmed_abstracts(pmids: list[str]) -> str:
    """Fetch titles and abstracts for a list of PubMed IDs.

    Use this after search_pubmed to assess whether the papers contain relevant
    disease-to-target evidence. Input must be a list of PMIDs returned by
    search_pubmed.
    """
    if not pmids:
        return "No PMIDs were provided, so no abstracts could be fetched."

    try:
        response = requests.get(
            f"{NCBI_BASE_URL}/efetch.fcgi",
            params={
                **_ncbi_parameters(),
                "db": "pubmed",
                "id": ",".join(pmids),
                "retmode": "xml",
            },
            timeout=20,
        )
        response.raise_for_status()
        root = ET.fromstring(response.content)

        papers = []
        for article in root.findall(".//PubmedArticle"):
            pmid = article.findtext(".//PMID")
            title_element = article.find(".//ArticleTitle")
            title = "".join(title_element.itertext()) if title_element is not None else "Untitled"

            abstract_parts = [
                "".join(abstract_element.itertext()).strip()
                for abstract_element in article.findall(".//Abstract/AbstractText")
            ]
            abstract = " ".join(part for part in abstract_parts if part)

            papers.append(
                {
                    "pmid": pmid,
                    "title": title,
                    "abstract": abstract or "No abstract available",
                    "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                }
            )

        return json.dumps(papers, indent=2)
    except (requests.RequestException, ET.ParseError) as error:
        return f"PubMed abstract request failed: {error}. Do not invent abstract evidence."

In [ ]:
### Tool Uniprot.
@tool
def lookup_uniprot(
    gene_symbol: str | None = None,
    taxonomy: str | None = None,
    organism_id: int | None = None,
    keyword: str | None = None,
    protein_name: str | None = None,
    reviewed: bool | None = True,
    all_terms: str | None = None,
    advanced_query: str | None = None,
    max_results: int = 3,
) -> str:
    """Search UniProtKB with optional advanced-search filters.

    Use the named filters when the user provides them: gene_symbol (for example
    IL6), taxonomy (for example Homo sapiens), organism_id (for example 9606),
    keyword (for example Cytokine), protein_name, and reviewed status.

    all_terms searches across UniProtKB fields. advanced_query accepts a complete
    UniProt query expression when the user needs filters not listed here, for
    example: gene:IL6 AND organism_id:9606 AND reviewed:true.
    """
    def quote(value: str) -> str:
        return f'"{value.replace(chr(34), chr(92) + chr(34))}"'

    clauses = []
    if advanced_query:
        clauses.append(f"({advanced_query})")
    if gene_symbol:
        clauses.append(f"gene:{quote(gene_symbol)}")
    if taxonomy:
        clauses.append(f"organism_name:{quote(taxonomy)}")
    if organism_id is not None:
        clauses.append(f"organism_id:{organism_id}")
    if keyword:
        clauses.append(f"keyword:{quote(keyword)}")
    if protein_name:
        clauses.append(f"protein_name:{quote(protein_name)}")
    if all_terms:
        clauses.append(f"({all_terms})")
    if reviewed is not None:
        clauses.append(f"reviewed:{str(reviewed).lower()}")

    if not clauses:
        return "Provide at least one UniProt search parameter."

    try:
        response = requests.get(
            UNIPROT_URL,
            params={
                "query": " AND ".join(clauses),
                "format": "json",
                "fields": "accession,id,protein_name,gene_names,organism_name,cc_function",
                "size": max(1, min(max_results, 10)),
            },
            timeout=20,
        )
        response.raise_for_status()
        results = response.json().get("results", [])
        if not results:
            return "No UniProtKB records matched the supplied search parameters."

        proteins = []
        for item in results:
            description = item.get("proteinDescription", {})
            recommended = description.get("recommendedName", {}).get("fullName", {}).get("value")
            proteins.append(
                {
                    "accession": item.get("primaryAccession"),
                    "entry_name": item.get("uniProtkbId"),
                    "protein_name": recommended or "No recommended name returned",
                    "genes": [gene.get("geneName", {}).get("value") for gene in item.get("genes", [])],
                    "organism": item.get("organism", {}).get("scientificName"),
                    "url": f"https://www.uniprot.org/uniprotkb/{item.get('primaryAccession')}",
                }
            )
        return json.dumps(proteins, indent=2)
    except requests.RequestException as error:
        return f"UniProt request failed: {error}. Do not invent protein data."

In [ ]:
### Tools:-
TOOLS = [search_pubmed, fetch_pubmed_abstracts, lookup_uniprot]

In [ ]:
### Objective for LLM (what he reads and does in accordance of)
SYSTEM_PROMPT = """
You are a biomedical research-support agent for early disease-to-target exploration.

You must follow this workflow in order:

PHASE 1 — Literature collection
1. Use search_pubmed for the disease query.
2. Fetch PubMed abstracts with fetch_pubmed_abstracts.
3. Do not call lookup_uniprot during this phase.

PHASE 2 — Evidence filtering
4. Read every available PubMed abstract.
5. Retain only papers that directly link the disease to a biological target,
   pathway, mechanism, or therapeutic rationale.
6. Exclude papers with no abstract, weak relevance, or no target-related evidence.

PHASE 3 — Protein verification
7. Derive at most three candidate gene/protein targets only from the retained literature.
8. Call lookup_uniprot only for those candidates.
9. Produce a source-backed research-hypothesis report.

Never diagnose, recommend treatment, claim a target is clinically validated, or invent data.
Clearly label the output: 'Research hypothesis - requires expert validation.'
"""

In [ ]:
### Model thinks and decides whether to use tool or not.
def call_model(state: MessagesState) -> dict[str, list]:
    """Invoke the tool-bound model with the system prompt and current messages."""
    messages = [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}


if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY is missing. Copy .env.example to .env and add your key.")

In [ ]:
### calling the model (Note: Temperature = degree of randomness llm allows in results. 0=0 randomness)
model = ChatGroq(model="llama-3.1-8b-instant", temperature=1)
model_with_tools = model.bind_tools(TOOLS)

In [ ]:
### Graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", call_model)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
agent = builder.compile()

In [ ]:
### Output.
def run() -> None:
    print("Disease-to-Target Agent (research support only)")
    disease = input("Enter a disease (or type quit): ").strip()
    if not disease or disease.lower() == "quit":
        return

    result = agent.invoke(
        {"messages": [HumanMessage(content=f"Identify exploratory therapeutic targets for {disease}.")]}
    )
    print("\n" + result["messages"][-1].content)


if __name__ == "__main__":
    run()